# BELLHOP Advanced Scenarios

Advanced multi-scenario analyses built on top of the main interactive notebook.

## Contents
1. **SOFAR Channel Animation** — Source sinking through water column, animated ray plots
2. **Frequency Sweep** — TL comparison across 5 frequencies
3. **Range-Dependent Bathymetry** — Sloping continental shelf
4. **Communication Channel Simulator** — BPSK through acoustic channel + BER vs SNR


In [ ]:
import ipympl                                          # must be imported before matplotlib
%matplotlib widget
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

import src.bellhop_config as bc
bc.configure(verbose=True)
import arlpy.uwapm as pm

import src.profiles as prof
import src.environment_builder as eb
import src.plotting as pl
import src.utils as ut


---
## 1 — SOFAR Channel Animation

Step the source depth from near-surface to maximum depth.
For each step, BELLHOP computes ray paths and we assemble them into an animation
showing how propagation geometry evolves as the source sinks through the water column.


In [ ]:
# ── SOFAR animation controls ──────────────────────────────────────────────────
anim_water_depth  = 5000.0
anim_max_range    = 150.0   # km
anim_n_src_depths = 12      # number of animation frames
anim_freq         = 250.0
anim_bottom       = 'sand'

ssp_munk = prof.munk_profile(z_max=anim_water_depth)

# Source depths from near-surface to bottom
src_depths = np.linspace(10, anim_water_depth - 50, anim_n_src_depths)

print(f'Computing {anim_n_src_depths} ray frames (one per source depth)...')

frames_rays = []
for i, src_d in enumerate(src_depths):
    ssp_use = eb._clip_ssp(ssp_munk, anim_water_depth)
    env = pm.create_env2d(
        depth=anim_water_depth,
        soundspeed=ssp_use.tolist(),
        frequency=anim_freq,
        tx_depth=float(src_d),
        rx_depth=np.linspace(0, anim_water_depth, 31),
        rx_range=np.linspace(0.5, anim_max_range, 60),
        nbeams=80, min_angle=-60, max_angle=60,
    )
    eb._apply_bottom(env, anim_bottom)
    rays = pm.compute_rays(env)
    frames_rays.append((src_d, rays))
    print(f'  Frame {i+1}/{anim_n_src_depths}: src depth = {src_d:.0f} m')

print('All frames computed. Building animation...')


In [ ]:
# ── Build and display animation ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
max_range_m = anim_max_range * 1000.0

def draw_frame(frame_idx):
    ax.clear()
    src_d, rays = frames_rays[frame_idx]

    ax.set_facecolor('#ddeeff')
    ax.fill_between([0, max_range_m],
                    [anim_water_depth] * 2, [anim_water_depth * 1.06] * 2,
                    color='#8B6914', alpha=0.9)
    ax.axhline(anim_water_depth, color='#8B6914', linewidth=2)
    ax.axhline(0, color='#2196F3', linewidth=2)

    if rays is not None:
        cmap = plt.cm.RdYlBu
        n_r = len(rays)
        for i, (_, row) in enumerate(rays.iterrows()):
            if 'x' in row and 'y' in row:
                ax.plot(np.asarray(row['x']), np.asarray(row['y']),
                        color=cmap(i / max(n_r - 1, 1)), linewidth=0.7, alpha=0.6)

    ax.plot(0, src_d, 'r*', markersize=16, zorder=10)
    ax.set_xlim(0, max_range_m)
    ax.set_ylim(anim_water_depth * 1.06, -anim_water_depth * 0.02)
    ax.set_xlabel('Range (m)', fontsize=11)
    ax.set_ylabel('Depth (m)', fontsize=11)
    ax.set_title(
        f'SOFAR Channel Animation  |  Source at {src_d:.0f} m depth  |  '
        f'Frame {frame_idx+1}/{len(frames_rays)}',
        fontsize=12
    )
    ax.grid(True, alpha=0.2)

    # Overlay SOFAR axis
    sofar_d = prof.get_sofar_depth(ssp_munk)
    ax.axhline(sofar_d, color='yellow', linewidth=1.5, linestyle='--',
               alpha=0.8, label=f'SOFAR axis: {sofar_d:.0f} m')
    ax.legend(fontsize=9)

anim = animation.FuncAnimation(fig, draw_frame, frames=len(frames_rays),
                                interval=800, repeat=True)
plt.tight_layout()

# Render as HTML5 video
from IPython.display import HTML
HTML(anim.to_jshtml())


---
## 2 — Frequency Sweep

Higher frequencies: shorter wavelengths, sharper interference patterns, higher absorption.
Lower frequencies: longer range, smoother TL, but less directional.

We sweep from 100 Hz to 5 kHz for the same Munk-profile deep-ocean environment.


In [ ]:
# ── Frequency sweep ───────────────────────────────────────────────────────────
sweep_freqs  = [100, 500, 1000, 2500, 5000]   # Hz
sweep_depth  = 5000.0    # m
sweep_src    = 100.0     # m
sweep_range  = 100.0     # km
sweep_bottom = 'sand'

ssp_munk = prof.munk_profile(z_max=sweep_depth)
ssp_use  = eb._clip_ssp(ssp_munk, sweep_depth)

tl_results = {}
rx_depths  = np.linspace(0, sweep_depth, 51)
rx_ranges  = np.linspace(0.5, sweep_range, 80)

for freq in sweep_freqs:
    env = pm.create_env2d(
        depth=sweep_depth,
        soundspeed=ssp_use.tolist(),
        frequency=float(freq),
        tx_depth=sweep_src,
        rx_depth=rx_depths,
        rx_range=rx_ranges,
        nbeams=200,
    )
    eb._apply_bottom(env, sweep_bottom)
    tl = pm.compute_transmission_loss(env, mode=pm.incoherent)
    tl_results[freq] = tl
    print(f'  f={freq} Hz complete')

print('Frequency sweep done.')


In [ ]:
# ── Plot frequency sweep results ─────────────────────────────────────────────
fig, axes = plt.subplots(len(sweep_freqs), 1, figsize=(14, 4*len(sweep_freqs)),
                          sharex=True)

for i, freq in enumerate(sweep_freqs):
    tl = tl_results[freq]
    if tl is None:
        axes[i].set_title(f'{freq} Hz — no data')
        continue
    depths = np.asarray(tl.index, dtype=float)
    ranges_km = np.asarray(tl.columns, dtype=float)
    tl_matrix = np.abs(np.asarray(tl.values, dtype=float))
    tl_matrix = np.where(tl_matrix < 1, np.nan, tl_matrix)
    vmin = np.nanmin(tl_matrix)
    im = axes[i].pcolormesh(ranges_km, depths, tl_matrix,
                             vmin=vmin, vmax=vmin+70, cmap='jet', shading='auto')
    axes[i].fill_between(ranges_km[[0,-1]], [sweep_depth]*2, [sweep_depth*1.05]*2,
                          color='#8B6914', alpha=0.9)
    axes[i].axhline(sweep_depth, color='#8B6914', linewidth=2)
    axes[i].set_ylim(sweep_depth*1.05, -sweep_depth*0.01)
    axes[i].set_ylabel('Depth (m)', fontsize=10)
    axes[i].set_title(f'f = {freq} Hz  (incoherent TL)', fontsize=11)
    axes[i].grid(True, alpha=0.15, color='white')
    plt.colorbar(im, ax=axes[i], label='TL (dB)', fraction=0.025)

axes[-1].set_xlabel('Range (km)', fontsize=11)
plt.suptitle('Frequency Sweep — Transmission Loss vs Frequency\nMunk Profile, 5000 m depth',
             fontsize=13, fontweight='bold', y=1.002)
plt.tight_layout()
plt.show()


In [ ]:
# ── TL at source depth vs range for each frequency ───────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))
colors = plt.cm.plasma(np.linspace(0, 0.85, len(sweep_freqs)))

for i, freq in enumerate(sweep_freqs):
    tl = tl_results[freq]
    if tl is None: continue
    depths = np.asarray(tl.index, dtype=float)
    ranges_km = np.asarray(tl.columns, dtype=float)
    d_idx = int(np.argmin(np.abs(depths - sweep_src)))
    tl_slice = np.abs(np.asarray(tl.iloc[d_idx, :], dtype=float))
    ax.plot(ranges_km, tl_slice, color=colors[i], linewidth=2,
            label=f'{freq} Hz')

ax.invert_yaxis()
ax.set_xlabel('Range (km)', fontsize=11)
ax.set_ylabel('Transmission Loss (dB)', fontsize=11)
ax.set_title(f'TL at Source Depth ({sweep_src:.0f} m) vs Range — Frequency Comparison', fontsize=12)
ax.grid(True, alpha=0.25)
ax.legend(title='Frequency', fontsize=10)
plt.tight_layout()
plt.show()


---
## 3 — Range-Dependent Bathymetry

Model a continental shelf: the bottom slopes from shallow (200 m) to deep (3000 m)
over 80 km — a typical continental slope geometry. arlpy accepts a range-dependent
bathymetry as a list of (range_km, depth_m) pairs.

This shows how rays are refracted differently as the water deepens:
- On the shelf: strong bottom interaction, high TL
- At the slope: rays can escape to deep water and form convergence paths
- In the deep: SOFAR channel becomes accessible


In [ ]:
# ── Range-dependent bathymetry ────────────────────────────────────────────────
# Depth profile: shallow shelf (200 m) slopes to abyss (3500 m) over 80 km
bathy_ranges = np.array([0, 30, 60, 80, 120])   # km
bathy_depths = np.array([200, 250, 800, 2500, 3500])  # m

# Use a Munk-ish SSP but shallower for the shelf portion
ssp_full = prof.munk_profile(z_max=3500, n_points=200)

max_depth = float(bathy_depths.max())
ssp_use = eb._clip_ssp(ssp_full, max_depth)

env_rd = pm.create_env2d(
    depth=list(zip(bathy_ranges.tolist(), bathy_depths.tolist())),
    soundspeed=ssp_use.tolist(),
    frequency=500.0,
    tx_depth=50.0,   # shallow source — on the shelf
    rx_depth=np.linspace(0, max_depth, 51),
    rx_range=np.linspace(0.2, 120.0, 100),
    nbeams=200,
    min_angle=-70, max_angle=70,
)
eb._apply_bottom(env_rd, 'sand')

rays_rd = pm.compute_rays(env_rd)
tl_rd   = pm.compute_transmission_loss(env_rd, mode=pm.incoherent)

print('Range-dependent scenario complete.')
print(f'  Rays: {len(rays_rd) if rays_rd is not None else "None"}')
print(f'  TL shape: {tl_rd.shape if tl_rd is not None else "None"}')


In [ ]:
# ── Plot range-dependent results ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Ray diagram with sloped bottom
ax_r = axes[0]
from scipy.interpolate import interp1d
bathy_interp = interp1d(bathy_ranges, bathy_depths, kind='linear', fill_value='extrapolate')
range_m = np.linspace(0, 120000, 300)
depth_at_range = bathy_interp(range_m / 1000.0)

ax_r.set_facecolor('#ddeeff')
ax_r.fill_between(range_m, depth_at_range, depth_at_range * 1.08,
                   color='#8B6914', alpha=0.9, label='Bottom')
ax_r.plot(range_m, depth_at_range, color='#8B6914', linewidth=2)
ax_r.axhline(0, color='#2196F3', linewidth=2)

if rays_rd is not None:
    cmap = plt.cm.RdYlBu
    n_r = len(rays_rd)
    for i, (_, row) in enumerate(rays_rd.iterrows()):
        if 'x' in row and 'y' in row:
            ax_r.plot(np.asarray(row['x']), np.asarray(row['y']),
                      color=cmap(i / max(n_r-1, 1)), linewidth=0.7, alpha=0.55)

ax_r.plot(0, 50, 'r*', markersize=16, zorder=10, label='Source (50 m, shelf)')
ax_r.set_xlim(0, 120000)
ax_r.set_ylim(max_depth * 1.08, -max_depth * 0.02)
ax_r.set_xlabel('Range (m)', fontsize=11)
ax_r.set_ylabel('Depth (m)', fontsize=11)
ax_r.set_title('Ray Diagram — Continental Slope Bathymetry | f=500 Hz', fontsize=12)
ax_r.grid(True, alpha=0.2)
ax_r.legend(fontsize=9)

# TL map
if tl_rd is not None:
    depths = np.asarray(tl_rd.index, dtype=float)
    ranges_km = np.asarray(tl_rd.columns, dtype=float)
    tl_mat = np.abs(np.asarray(tl_rd.values, dtype=float))
    tl_mat = np.where(tl_mat < 1, np.nan, tl_mat)
    vmin = np.nanmin(tl_mat)
    im = axes[1].pcolormesh(ranges_km, depths, tl_mat,
                             vmin=vmin, vmax=vmin+70, cmap='jet', shading='auto')
    # Overlay bathymetry line
    axes[1].plot(bathy_ranges, bathy_depths, color='white', linewidth=2, label='Bottom')
    axes[1].set_ylim(max_depth * 1.08, -max_depth * 0.02)
    axes[1].set_xlabel('Range (km)', fontsize=11)
    axes[1].set_ylabel('Depth (m)', fontsize=11)
    axes[1].set_title('Transmission Loss — Continental Slope', fontsize=12)
    axes[1].grid(True, alpha=0.15, color='white')
    axes[1].legend(fontsize=9)
    plt.colorbar(im, ax=axes[1], label='TL (dB)', fraction=0.025)

plt.tight_layout()
plt.show()


---
## 4 — Communication Channel Simulator

Use the BELLHOP channel impulse response to simulate an underwater acoustic
communication link:

1. Generate a BPSK symbol sequence
2. Convolve with the acoustic multipath channel
3. Add AWGN at a specified SNR
4. Apply a zero-forcing equalizer
5. Plot constellation and compute BER vs SNR

This directly connects ray acoustics output to the communications metrics
studied in the V2V OFDM project.


In [ ]:
# ── Compute arrivals for comms simulation ─────────────────────────────────────
comm_water_depth = 1000.0
comm_src_depth   = 50.0
comm_rx_range    = 15.0   # km
comm_rx_depth    = 100.0
comm_freq        = 3500.0   # Hz — narrowband carrier
comm_fs          = 10000.0  # sample rate

ssp_sw = prof.shallow_water_profile(total_depth=comm_water_depth)
ssp_sw_use = eb._clip_ssp(ssp_sw, comm_water_depth)

env_c = pm.create_env2d(
    depth=comm_water_depth,
    soundspeed=ssp_sw_use.tolist(),
    frequency=comm_freq,
    tx_depth=comm_src_depth,
    rx_depth=np.array([comm_rx_depth]),
    rx_range=np.array([comm_rx_range]),
    nbeams=1000,
)
eb._apply_bottom(env_c, 'sand')
arr_c = pm.compute_arrivals(env_c)

if arr_c is not None:
    ir = ut.channel_to_impulse_response(arr_c, fs=comm_fs)
    stats = ut.compute_arrival_stats(arr_c)
    print(f'Arrivals: {stats["n_arrivals"]}')
    print(f'Delay spread: {stats["delay_spread_ms"]:.2f} ms')
    print(f'Coherence BW: {stats["coherence_bandwidth_hz"]:.1f} Hz')
    print(f'IR length: {len(ir)} samples at {comm_fs:.0f} Hz')
else:
    print('No arrivals — using simple 3-path synthetic channel')
    ir = np.array([1.0, 0, 0, 0, 0.4, 0, 0, 0.2], dtype=complex)
    stats = {'n_arrivals': 3, 'delay_spread_ms': 0.7,
             'coherence_bandwidth_hz': 1429.0}


In [ ]:
# ── BPSK through acoustic channel ─────────────────────────────────────────────
from scipy.signal import convolve

np.random.seed(42)
n_symbols    = 2000
sps          = 4    # samples per symbol
symbol_rate  = comm_fs / sps

# Generate BPSK symbols: ±1
bits    = np.random.randint(0, 2, n_symbols)
symbols = 2 * bits - 1   # map 0→-1, 1→+1

# Upsample (NRZ pulse shaping)
tx_signal = np.repeat(symbols.astype(float), sps)

# Normalize IR
ir_norm = ir / (np.abs(ir).max() + 1e-12)

def simulate_link(snr_db):
    """Convolve tx through channel, add AWGN, detect, return BER."""
    rx_noisy = convolve(tx_signal, np.real(ir_norm), mode='full')[:len(tx_signal)]
    sig_power = np.mean(rx_noisy**2)
    noise_power = sig_power / (10**(snr_db/10))
    noise = np.sqrt(noise_power) * np.random.randn(len(rx_noisy))
    rx = rx_noisy + noise
    # Downsample and detect
    rx_down = rx[sps//2::sps][:n_symbols]
    detected = (rx_down >= 0).astype(int)
    ber = np.mean(detected != bits[:len(detected)])
    return ber, rx_down

# BER vs SNR curve
snr_range = np.arange(-2, 22, 2)
bers = []
for snr in snr_range:
    ber_vals = [simulate_link(snr)[0] for _ in range(3)]
    bers.append(np.mean(ber_vals))

# Theoretical AWGN BPSK BER
from scipy.special import erfc
ber_awgn = 0.5 * erfc(np.sqrt(10**(snr_range/10)))

print(f'BER simulation complete over SNR range {snr_range[0]}–{snr_range[-1]} dB')


In [ ]:
# ── Plot comms simulation results ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# 1: Channel IR
ax_ir = axes[0]
t_ms = np.arange(len(ir)) / comm_fs * 1000.0
ax_ir.stem(t_ms, np.abs(ir_norm), linefmt='#1a6fa3', markerfmt='o', basefmt='k-')
ax_ir.set_xlabel('Time (ms)', fontsize=11)
ax_ir.set_ylabel('Amplitude', fontsize=11)
ax_ir.set_title(
    f'Channel Impulse Response\n'
    f'{stats["n_arrivals"]} paths  |  '
    f'Δτ = {stats["delay_spread_ms"]:.2f} ms  |  '
    f'fs = {comm_fs:.0f} Hz',
    fontsize=11
)
ax_ir.grid(True, alpha=0.25)

# 2: Constellation at mid-SNR
_, rx_down_10 = simulate_link(10)
ax_con = axes[1]
colors_bits = ['#e74c3c' if b==0 else '#2980b9' for b in bits[:len(rx_down_10)]]
ax_con.scatter(rx_down_10, np.zeros(len(rx_down_10)), c=colors_bits,
               alpha=0.3, s=10)
ax_con.axvline(0, color='k', linewidth=2)
ax_con.set_xlabel('Decision Variable', fontsize=11)
ax_con.set_ylabel('Q (imaginary)', fontsize=11)
ax_con.set_title('BPSK Constellation  |  SNR = 10 dB\n(red=0, blue=1)', fontsize=11)
ax_con.set_xlim(-3, 3)
ax_con.grid(True, alpha=0.25)

# 3: BER vs SNR
ax_ber = axes[2]
ax_ber.semilogy(snr_range, np.maximum(bers, 1e-5), 'o-', color='#e67e22',
                linewidth=2, label='BELLHOP channel (BPSK)')
ax_ber.semilogy(snr_range, ber_awgn, 'k--', linewidth=1.5, label='Ideal AWGN BPSK')
ax_ber.set_xlabel('SNR (dB)', fontsize=11)
ax_ber.set_ylabel('Bit Error Rate', fontsize=11)
ax_ber.set_title('BER vs SNR — Multipath vs AWGN', fontsize=12)
ax_ber.grid(True, alpha=0.25, which='both')
ax_ber.legend(fontsize=10)
ax_ber.set_ylim(1e-4, 1)
ax_ber.axhline(1e-3, color='green', linestyle=':', alpha=0.6, label='10⁻³ target')

plt.suptitle(
    f'BPSK Through Acoustic Multipath Channel\n'
    f'BELLHOP: {comm_water_depth:.0f}m depth, {comm_rx_range:.0f}km range, '
    f'{comm_freq:.0f}Hz, f_s={comm_fs:.0f}Hz',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.show()

isi_taps = int(stats["delay_spread_ms"]/1000 * comm_fs / sps)
print(f'\nISI length: {isi_taps} symbols  (equalizer must span at least this many taps)')
print(f'Coherence BW: {stats["coherence_bandwidth_hz"]:.1f} Hz')
print(f'Symbol rate: {symbol_rate:.0f} symbols/sec')
print(f'Symbol period: {1/symbol_rate*1000:.2f} ms')
print(f'Guard interval needed (OFDM CP): ≥ {stats["delay_spread_ms"]:.2f} ms')
